In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q11/annotated/checkpoints/pre_cell_3.pickle

In [ ]:
%%RecordEvent
%%time
### cell 3 ###

# Compute total cost and select required columns in one step
ps = partsupp.assign(
    TOTAL_COST=partsupp.PS_SUPPLYCOST * partsupp.PS_AVAILQTY
)[["PS_PARTKEY", "PS_SUPPKEY", "TOTAL_COST"]]

# Filter supplier and nation tables
sup = supplier[["S_SUPPKEY", "S_NATIONKEY"]]
nat = nation[nation.N_NAME == "GERMANY"][["N_NATIONKEY"]]

# Join partsupp → supplier → nation, then keep only PS_PARTKEY and TOTAL_COST
df = (
    ps
    .merge(sup, on="PS_SUPPKEY", how="inner")
    .merge(nat, left_on="S_NATIONKEY", right_on="N_NATIONKEY", how="inner")
    [["PS_PARTKEY", "TOTAL_COST"]]
)

# Compute threshold and aggregate per part
threshold = df.TOTAL_COST.sum() * 0.0001

total = (
    df
    .groupby("PS_PARTKEY")["TOTAL_COST"]
    .sum()
    .reset_index()
    .rename(columns={"TOTAL_COST": "VALUE"})
)

# Filter and sort
total = total[total.VALUE > threshold].sort_values("VALUE", ascending=False)

In [ ]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q11/rewritten/o4_mini_high/checkpoints/post_cell_3_try_2.pickle

In [ ]:
%PrintCellInfo opt_cell_exec_info

In [ ]:

with open("/home/dias-benchmarks/tpch/notebooks/q11/opt_cell_exec_info_3_try_2.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[3], f)


In [ ]:
opt_output = Out.get(4)